# Behaviour Dataset Processing

## Overview

This notebook processes a single National Travel Survey (NTS) behavioural dataset.

The objective is to clean, harmonise, and transform mobility behaviour information into a standardised behavioural representation for the proposed representative population framework.

The processed behavioural data will be exported as a standardised JSON file for subsequent integration with the demographic layer.

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset | NTS0409: Average number of trips and distance travelled by purpose and main mode: England, 2002 onwards |
| Source |National Travel Survey|
| Geography | England |
| Input File |nts0409.ods|
| Output File |nts0409_travel_mode_probability.json|
| Notebook Author | Peter |
| Last Updated | 2026-07-14 |

## 1. Import

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## 2. Configuration

In [2]:
from pathlib import Path

# File paths

input_file = "nts0409.ods"
output_file = "nts0409_travel_mode_probability.json"

raw_data_path = Path("../../data/nts/raw")
output_data_path = Path("../../data/nts/harmonised")

## 3. Load Dataset

In [3]:
df = pd.read_excel(
    raw_data_path / input_file,
    sheet_name="NTS0409a_trips",
    skiprows=5
)

df_processed = df.copy()

df_processed.head()

,Year,Main mode,Commuting,Business,Education or escort education,Shopping,Other escort,Personal business,Leisure [note 2],Just walk [note 8],Other [note 8],All purposes,Unweighted sample size: trips (thousands)
0,2002,"Walk [notes 3, 4]",16.349172,2.865760,44.115380,59.249569,13.319239,34.740433,52.424669,40.874737,0.073657,264.012617,66.356
1,2002,Pedal cycle [note 5],6.547287,0.612595,1.833587,2.363763,0.364849,1.308123,4.999214,0.000000,0.000000,18.029417,4.542
2,2002,Car or van driver,95.832523,26.226708,23.895374,90.122423,57.491243,47.131800,97.393661,0.000000,0.320400,438.414132,116.736
3,2002,Car or van passenger,16.985655,2.400159,25.759621,46.019618,30.738702,24.580872,93.407171,0.000000,0.043487,239.935284,63.814
4,2002,Motorcycle,2.005321,0.041155,0.051922,0.471992,0.114125,0.297173,0.864034,0.000000,0.006289,3.852011,0.977


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [4]:
df_processed.info()

print(f"Shape: {df_processed.shape}")

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 13 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Year                                       322 non-null    int64  
 1   Main mode                                  322 non-null    object 
 2   Commuting                                  322 non-null    float64
 3   Business                                   322 non-null    float64
 4   Education or escort education              322 non-null    float64
 5   Shopping                                   322 non-null    float64
 6   Other escort                               322 non-null    float64
 7   Personal business                          322 non-null    float64
 8   Leisure [note 2]                           322 non-null    float64
 9   Just walk [note 8]                         322 non-null    float64
 10  Other [note 8]            

Index(['Year', 'Main mode', 'Commuting', 'Business',
       'Education or escort education', 'Shopping', 'Other escort',
       'Personal business', 'Leisure [note 2]', 'Just walk [note 8]',
       'Other [note 8]', 'All purposes',
       'Unweighted sample size: trips (thousands)'],
      dtype='object')

## 5. Data Cleaning

Check the dataset quality before feature processing.

In [5]:
# Missing values
df_processed.isna().sum()

Year                                         0
Main mode                                    0
Commuting                                    0
Business                                     0
Education or escort education                0
Shopping                                     0
Other escort                                 0
Personal business                            0
Leisure [note 2]                             0
Just walk [note 8]                           0
Other [note 8]                               0
All purposes                                 0
Unweighted sample size: trips (thousands)    0
dtype: int64

In [6]:
# Duplicate records
df_processed.duplicated().sum()

np.int64(0)

In [7]:
# Summary statistics
df_processed.describe(include="all")

,Year,Main mode,Commuting,Business,Education or escort education,Shopping,Other escort,Personal business,Leisure [note 2],Just walk [note 8],Other [note 8],All purposes,Unweighted sample size: trips (thousands)
count,322.000000,322,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000
unique,NaN,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,"Walk [notes 3, 4]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2013.000000,NaN,20.428585,4.111453,16.513472,26.732978,12.369668,13.385157,35.643454,8.257334,0.062890,137.504991,37.491876
std,6.643574,NaN,40.062141,8.778539,30.871271,50.876966,24.788036,25.558970,66.987845,21.875874,0.150439,257.872228,72.540134
min,2002.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.100548,0.000000,0.000000,0.104140,0.012000
25%,2007.000000,NaN,1.069626,0.192265,0.489498,0.601438,0.092821,0.508234,1.880841,0.000000,0.000000,6.689651,1.255250
50%,2013.000000,NaN,4.918276,0.500114,2.006025,1.629908,0.347983,1.079687,4.838842,0.000000,0.000000,16.170226,4.273500
75%,2019.000000,NaN,12.182254,1.913923,23.143866,36.054863,10.763137,20.041321,41.916900,0.000000,0.007191,208.690574,53.497750


## 6. Feature Processing

Process the behavioural information required for the representative population framework.

The processing workflow in this section depends on the specific NTS dataset.

In [8]:
df_processed = df_processed[
    df_processed['Year'] == 2024].copy()

In [9]:
# melt means to transform the data from wide format to long format. 
# In this case, we are transforming the data so that each row represents a single trip purpose and its corresponding number of trips for each main mode of transport in the year 2024.
df_processed = df_processed.melt(
    id_vars=['Year', 'Main mode'],
    value_vars=[col for col in df_processed.columns if col not in ['Year', 'Main mode']],
    var_name="Purpose",
    value_name="Trips"
)

In [10]:
df_processed = df_processed[
    ~df_processed['Purpose'].isin([
        'All purposes',
        'Unweighted sample size: trips (thousands)'
    ])
]

df_processed = df_processed[
    df_processed['Main mode'] != 'All modes'
]

df_processed['Probability'] = (
    df_processed['Trips'] /
    df_processed.groupby('Purpose')['Trips'].transform('sum')
)

In [11]:
df_processed.head()

,Year,Main mode,Purpose,Trips,Probability
0,2024,"Walk [notes 3, 4]",Commuting,14.090618,0.126616
1,2024,Pedal cycle [note 5],Commuting,4.657803,0.041854
2,2024,Car or van driver,Commuting,60.443130,0.543133
3,2024,Car or van passenger,Commuting,8.244171,0.074081
4,2024,Motorcycle,Commuting,1.037418,0.009322


In [12]:
#verify the sum of probabilities for each purpose equals 1
for purpose in df_processed['Purpose'].unique():
    total_probability = df_processed[df_processed['Purpose'] == purpose]['Probability'].sum()
    print(f"Total probability for {purpose}: {total_probability}")

Total probability for Commuting: 0.9999999999999999
Total probability for Business: 1.0000000000000002
Total probability for Education or escort education: 1.0
Total probability for Shopping: 1.0
Total probability for Other escort: 1.0000000000000002
Total probability for Personal business: 0.9999999999999999
Total probability for Leisure [note 2]: 1.0
Total probability for Just walk [note 8]: 1.0
Total probability for Other [note 8]: 0.9999999999999999


In [13]:
nts0409a_matrix = df_processed.pivot(
    index = 'Purpose',
    columns = 'Main mode',
    values = 'Probability'
)

In [14]:
nts0409a_matrix

Main mode,Bus in London,Car or van driver,Car or van passenger,London Underground,Motorcycle,Non-local bus,Other local bus,Other private transport [note 6],Other public transport [note 7],Pedal cycle [note 5],Surface Rail,Taxi or minicab,"Walk [notes 3, 4]"
Purpose,,,,,,,,,,,,,
Business,0.014149,0.686180,0.058015,0.021660,0.001818,0.000000,0.022984,0.007440,0.002132,0.018543,0.068402,0.011831,0.086845
Commuting,0.025553,0.543133,0.074081,0.036511,0.009322,0.000383,0.053567,0.004601,0.004547,0.041854,0.069340,0.010494,0.126616
Education or escort education,0.020506,0.234031,0.231633,0.005851,0.000177,0.000000,0.046675,0.025720,0.001263,0.011719,0.011596,0.006177,0.404653
Just walk [note 8],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
Leisure [note 2],0.012548,0.379552,0.322847,0.012225,0.002348,0.001344,0.024635,0.008854,0.002579,0.024577,0.029235,0.016470,0.162787
Other [note 8],0.016808,0.843711,0.113766,0.000000,0.000000,0.000000,0.025715,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Other escort,0.004971,0.564046,0.282615,0.000886,0.000208,0.000000,0.013024,0.007613,0.000909,0.002111,0.003210,0.004454,0.115954
Personal business,0.016379,0.425456,0.222890,0.006716,0.000799,0.000200,0.031427,0.008118,0.001792,0.009595,0.011557,0.014737,0.250334
Shopping,0.015155,0.457349,0.177411,0.003057,0.001244,0.000028,0.036393,0.004690,0.001643,0.008853,0.007577,0.005078,0.281521


## 7. Validation

Validate the processed dataset before exporting.

In [15]:
for purpose in nts0409a_matrix.index:
    total_probability = nts0409a_matrix.loc[purpose].sum()
    print(f"Total probability for {purpose}: {total_probability}")

Total probability for Business: 1.0
Total probability for Commuting: 0.9999999999999999
Total probability for Education or escort education: 1.0
Total probability for Just walk [note 8]: 1.0
Total probability for Leisure [note 2]: 1.0
Total probability for Other [note 8]: 0.9999999999999999
Total probability for Other escort: 1.0000000000000002
Total probability for Personal business: 1.0000000000000002
Total probability for Shopping: 1.0


In [16]:
if np.all(np.isclose(nts0409a_matrix.sum(axis=1), 1.0)):
    print("All travel mode probability distributions sum to 1.")
else:
    print("Some rows do not sum to 1.")

All travel mode probability distributions sum to 1.


## 8. Export

Export the processed dataset for later integration.

In [17]:
print(nts0409a_matrix.index)

Index(['Business', 'Commuting', 'Education or escort education',
       'Just walk [note 8]', 'Leisure [note 2]', 'Other [note 8]',
       'Other escort', 'Personal business', 'Shopping'],
      dtype='object', name='Purpose')


In [18]:
nts0409a_matrix.rename(
    index={
        "Personal business": "Personal Business",
        "Education or escort education": "Education or Escort Education",
        "Other escort": "Other Escort",
        "Just walk [note 8]": "Just Walk",
        "Leisure [note 2]": "Leisure",
        "Other [note 8]": "Other"
    },
    inplace=True
)

In [ ]:
nts0409a_matrix = nts0409a_matrix.rename(columns={
    "Other private transport [note 6]": "Other Private Transport",
    "Other public transport [note 7]": "Other Public Transport",
    "Pedal cycle [note 5]": "Pedal Cycle",
    "Walk [notes 3, 4]": "Walk"
})

In [20]:
output_data_path.mkdir(parents=True, exist_ok=True)

nts0409a_export = nts0409a_matrix.reset_index()
nts0409a_export.to_dict("records")

nts0409a_export.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## 9. Summary

### Input

- National Travel Survey (NTS)

### Output

- Standardised behavioural JSON

### Processing Completed

- Dataset cleaned
- Behaviour processed
- Validation passed
- JSON exported